#### 文本流输出

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from agent_lab.model_hub import LLM_FAST, LLM_THINKING

model = init_chat_model(**LLM_FAST)

In [2]:
full = None
for chunk in model.stream('what color is the sky.'):
    full = chunk if full is None else full + chunk
    print(full.text)


That
That's
That's a
That's a great
That's a great question
That's a great question.


That's a great question.

The
That's a great question.

The simplest
That's a great question.

The simplest answer
That's a great question.

The simplest answer is
That's a great question.

The simplest answer is **
That's a great question.

The simplest answer is **blue
That's a great question.

The simplest answer is **blue**.


That's a great question.

The simplest answer is **blue**.

However
That's a great question.

The simplest answer is **blue**.

However,
That's a great question.

The simplest answer is **blue**.

However, the
That's a great question.

The simplest answer is **blue**.

However, the full
That's a great question.

The simplest answer is **blue**.

However, the full answer
That's a great question.

The simplest answer is **blue**.

However, the full answer is
That's a great question.

The simplest answer is **blue**.

However, the full answer is a
That's a great question.

Th

In [3]:
type(full), type(chunk)

(langchain_core.messages.ai.AIMessageChunk,
 langchain_core.messages.ai.AIMessageChunk)

In [4]:
from langchain.messages import AIMessageChunk

assert isinstance(full, AIMessageChunk)
assert isinstance(chunk, AIMessageChunk)

full.content_blocks

[{'type': 'text',
  'text': "That's a great question.\n\nThe simplest answer is **blue**.\n\nHowever, the full answer is a bit more interesting:\n\n*   **On a clear, sunny day:** The sky appears blue because of a phenomenon called **Rayleigh scattering**. Sunlight is made up of all colors. As it passes through the Earth's atmosphere, the blue light (which has shorter waves) is scattered in all directions by the gases and particles in the air more than the other colors. So, we see blue light coming from all over the sky.\n\n*   **At sunrise and sunset:** The sky often turns shades of **red, orange, and pink**. This is because the sun is lower in the sky, and its light has to travel through much more atmosphere. The blue light gets scattered away so much that it's gone, leaving the longer wavelength red and orange light to reach our eyes.\n\n*   **On a cloudy day:** The sky looks **gray or white**. The water droplets in clouds are much larger than air molecules. They scatter all waveleng

#### 工具调用流输出

In [ ]:
reasoner = init_chat_model(**LLM_THINKING)

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city"""
    return f"It's always sunny in {city}."

reasoner = reasoner.bind_tools([get_weather_for_location])

In [6]:
full = None
for chunk in reasoner.stream('what is the weather in sf?'):
    for block in chunk.content_blocks:
        if block['type'] == 'reasoning' and (reasoning := block.get('reasoning')):
            print(reasoning, end='')
        elif block['type'] == 'tool_call_chunk':
            print(block)
        elif block['type'] == 'text':
            print(block['text'], end='')
    full = chunk if full is None else full + chunk

The user is asking about the weather in SF, which likely means San Francisco. Let me get the weather for San Francisco.{'type': 'tool_call_chunk', 'id': 'call_00_U8o8PtZVT7xReOZRuoZcOHHQ', 'name': 'get_weather_for_location', 'args': '', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '{', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '"', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': 'city', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '"', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': ': ', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '"', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': 'San', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': ' Francisco', 'index': 0}
{'type': 'tool_call_chunk', 'id': None, 'name': None, 'args': '"', 'index': 0}
{'type': 'tool_call_chunk

In [7]:
type(full)

langchain_core.messages.ai.AIMessageChunk

In [8]:
assert isinstance(full, AIMessageChunk)

full.content_blocks

[{'type': 'reasoning',
  'reasoning': 'The user is asking about the weather in SF, which likely means San Francisco. Let me get the weather for San Francisco.'},
 {'type': 'tool_call',
  'id': 'call_00_U8o8PtZVT7xReOZRuoZcOHHQ',
  'name': 'get_weather_for_location',
  'args': {'city': 'San Francisco'}}]

#### 流式事件

In [9]:
async for event in model.astream_events("Hello"):

    if event["event"] == "on_chat_model_start":
        print(f"Input: {event['data']['input']}")

    elif event["event"] == "on_chat_model_stream":
        print(f"Token: {event['data']['chunk'].text}")

    elif event["event"] == "on_chat_model_end":
        print(f"Full message: {event['data']['output'].text}")

    else:
        pass

Input: Hello
Token: 
Token: 你好
Token: ！
Token: 有什么
Token: 可以
Token: 帮
Token: 你的
Token: 吗
Token: ？
Token: 😊
Token: 


Token: 请
Token: 随时
Token: 提出
Token: 你的
Token: 问题
Token: 或
Token: 需求
Token: 。
Token: 
Token: 
Full message: 你好！有什么可以帮你的吗？😊

请随时提出你的问题或需求。
